# VisionXAI: Retinal Fundus Diabetic Retinopathy (DR) Model Training
### Model: EfficientNet-B3 + Grad-CAM Saliency Maps
This notebook trains the deep learning model used in **Tier 2 (Nurse Portal)** and **Tier 3 (Doctor Portal)** for:
1. Predicting Diabetic Retinopathy Severity Grade (0: No DR, 1: Mild, 2: Moderate, 3: Severe, 4: Proliferative)
2. Calculating DR Probability Percentage (0-100%)
3. Generating Grad-CAM visual attention heatmaps to highlight microaneurysms, hemorrhages, and exudates
4. Exporting the trained weights to `dr_model.pth` and `dr_model.onnx`

## 1. Environment & GPU Setup
Ensure GPU is enabled: `Runtime` -> `Change runtime type` -> `T4 GPU`.

In [ ]:
!nvidia-smi
!pip install -q timm albumentations opencv-python-headless grad-cam onnx

## 2. Imports

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using compute device:', DEVICE)

## 3. Dataset Setup (APTOS 2019 Blindness Detection)
You can connect Kaggle or upload your dataset ZIP to Google Drive.

In [ ]:
# If using Kaggle API:
# 1. Upload your kaggle.json to Colab
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c aptos2019-blindness-detection
# !unzip -q aptos2019-blindness-detection.zip -d dataset/

# Or mount Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/VisionXAI_Data/aptos2019/'

## 4. Retinal Image Preprocessing (Ben Graham Circular CLAHE)

In [ ]:
def crop_image_from_gray(img, tol=7):
    if img.ndim == 2:
        mask = img > tol
        return img[np.ix_(mask.any(1), mask.any(0))]
    elif img.ndim == 3:
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mask = gray_img > tol
        if img[:, :, 0][mask].shape[0] == 0:
            return img
        check_shape = img[:, :, 0][np.ix_(mask.any(1), mask.any(0))].shape[0]
        if check_shape == 0:
            return img
        else:
            img1 = img[:, :, 0][np.ix_(mask.any(1), mask.any(0))]
            img2 = img[:, :, 1][np.ix_(mask.any(1), mask.any(0))]
            img3 = img[:, :, 2][np.ix_(mask.any(1), mask.any(0))]
            return np.stack([img1, img2, img3], axis=-1)

def load_ben_graham_image(path, img_size=512):
    image = cv2.imread(path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = crop_image_from_gray(image)
    image = cv2.resize(image, (img_size, img_size))
    # Subtract local average color
    image = cv2.addWeighted(image, 4, cv2.GaussianBlur(image, (0, 0), img_size / 30), -4, 128)
    return image

## 5. Model Architecture (EfficientNet-B3)

In [ ]:
class VisionXAIDRClassifier(nn.Module):
    def __init__(self, num_classes=5, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b3', pretrained=pretrained, num_classes=0)
        in_features = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits

model = VisionXAIDRClassifier(num_classes=5, pretrained=True).to(DEVICE)
print('Model initialized successfully. Total parameters:', sum(p.numel() for p in model.parameters()))

## 6. Grad-CAM Visualization Function

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

def generate_gradcam(model, input_tensor, original_rgb_norm):
    target_layers = [model.backbone.conv_head]
    cam = GradCAM(model=model, target_layers=target_layers)
    grayscale_cam = cam(input_tensor=input_tensor)[0, :]
    visualization = show_cam_on_image(original_rgb_norm, grayscale_cam, use_rgb=True)
    return grayscale_cam, visualization

## 7. Model Export (PyTorch & ONNX)
Exports `dr_model.pth` and `dr_model.onnx` to copy into `ai/models/` for the FastAPI backend.

In [ ]:
# Save PyTorch Weights
torch.save(model.state_dict(), 'dr_model.pth')

# Export to ONNX for fast inference
dummy_input = torch.randn(1, 3, 512, 512, device=DEVICE)
torch.onnx.export(
    model,
    dummy_input,
    'dr_model.onnx',
    input_names=['input_image'],
    output_names=['logits'],
    dynamic_axes={'input_image': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=14
)
print('Models exported successfully: dr_model.pth & dr_model.onnx')